In [0]:
# ============================================
# AUDIT LOGGER
# ============================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

from pyspark.sql.functions import (
    current_timestamp
)


def log_ingestion(
    file_name,
    target_table,
    record_count,
    status,
    audit_table,
    error_message=None
):

    schema = StructType([

        StructField(
            "file_name",
            StringType(),
            True
        ),

        StructField(
            "target_table",
            StringType(),
            True
        ),

        StructField(
            "record_count",
            IntegerType(),
            True
        ),

        StructField(
            "status",
            StringType(),
            True
        ),

        StructField(
            "error_message",
            StringType(),
            True
        )

    ])

    audit_df = spark.createDataFrame(

        [

            (
                str(file_name),
                str(target_table),
                int(record_count),
                str(status),
                error_message
            )

        ],

        schema=schema
    )

    audit_df = audit_df.withColumn(
        "load_timestamp",
        current_timestamp()
    )

    audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            audit_table
        )

In [0]:
# FILE PROCESS CHECK


def is_file_processed(
    file_name,
    audit_table
):

    count = spark.sql(f"""
    SELECT COUNT(*)
    FROM {audit_table}
    WHERE file_name = '{file_name}'
    AND status = 'SUCCESS'
    """).collect()[0][0]

    return count > 0